## RAG with Seismic

This notebook shows how to easily plug Seismic into your RAG pipeline based on Learned Sparse Representations first-stage. 

In [ ]:
import seismic
from seismic import SeismicIndex

#### Build you Seismic index from a jsonl file, where each line is a json object with the following format:

- id: must represent the ID of the document as an integer.
- content: the original content of the document, as a string. This field is optional.
- vector: a dictionary where each key represents a token, and its corresponding value is the score, e.g., {"dog": 2.45}.


Ensure to pass `load_content=True`.

In [ ]:
json_input_file = ""
index = SeismicIndex.build(
    json_input_file,
    n_postings=3000,
    centroid_fraction=0.1,
    min_cluster_size=2,
    summary_energy=0.4, 
    max_fraction=4.0,
    load_content=True)

##### Load the queries and search the index

In [ ]:
import numpy as np
import json

file_path = ""

queries = []
with open(file_path, 'r') as f:
    for line in f:
        queries.append(json.loads(line))

string_type = seismic.get_seismic_string()

queries_ids = np.array([q['id'] for q in queries], dtype=string_type)

query_components = []
query_values = []

for query in queries:
    vector = query['vector']
    query_components.append(np.array(list(vector.keys()), dtype=string_type))
    query_values.append(np.array(list(vector.values()), dtype=np.float32))
    
results = index.batch_search(
    queries_ids=queries_ids,
    query_components=query_components,
    query_values=query_values,
    k=10,
    query_cut=10,
    heap_factor=0.7,
    n_knn=0,
    sorted=True, 
    num_threads=1,
)

### Retrieve document texts for the top results 
#### Since the index was built with `load_content=True`, we can call `index.get_doc_text(doc_id)` to fetch the stored passage text.
#### Example: build a context prompt for the first query


In [ ]:

first_query_text = queries[0].get("text", queries[0].get("id", ""))
first_results = results[0]  # list of (query_id, score, doc_id) for query 0

context_passages = []
for query_id, score, doc_id in first_results:
    text = index.get_doc_text(doc_id)
    if text is not None:
        context_passages.append(f"[{doc_id}] {text}")

prompt = (
    f"Query: {first_query_text}\n\n"
    "Relevant passages:\n"
    + "\n\n".join(f"{i+1}. {p}" for i, p in enumerate(context_passages))
    + "\n\nAnswer based on the passages above:"
)

print(prompt)